# DriftGraph-BCP Implementation

Bayesian Change-Point-Guided Continual Graph Learning with Adaptive Conformal Prediction for Open-World IoT Intrusion Detection.

This notebook provides the end-to-end experimental workflow for DriftGraph-BCP. It covers data preparation, leakage-control audits, dynamic flow-similarity graph construction, continual GraphSAGE learning, online diagonal-Laplace posterior prediction, open-world evidence fusion, adaptive class-conditional RAPS, Bayesian online change-point monitoring, baseline and ablation suites, five-seed robustness, statistical testing, external replication, and manuscript artifact generation.

## 1. Project setup

The workflow assumes the CICIoT2023 CSV files are available locally and that an optional Edge-IIoTset folder is available for external replication. The default paths can be changed in the configuration cell.

In [ ]:
from __future__ import annotations

import os
import gc
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterable

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.covariance import LedoitWolf
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats

import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

try:
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv
    PYG_AVAILABLE = True
except Exception:
    PYG_AVAILABLE = False

warnings.filterwarnings('ignore')

In [ ]:
@dataclass
class ProjectConfig:
    project_name: str = 'DriftGraph-BCP'
    primary_dataset_name: str = 'CICIoT2023'
    external_dataset_name: str = 'Edge-IIoTset'
    primary_data_dir: str = './data/CICIoT2023'
    external_data_dir: str = './data/Edge-IIoTset'
    output_dir: str = './outputs/driftgraph_bcp'
    cache_dir: str = './cache/driftgraph_bcp'
    seed: int = 42
    robustness_seeds: Tuple[int, ...] = (42, 123, 202, 777, 999)
    feature_cap: int = 40
    knn_k: int = 12
    stream_snapshot_size: int = 3000
    snapshots_per_phase: int = 5
    pre_snapshots_per_task: int = 5
    post_snapshots_per_task: int = 5
    initial_attack_classes: int = 6
    arriving_classes_per_task: int = 2
    benign_stream_fraction: float = 0.30
    arriving_unknown_fraction: float = 0.18
    permanent_unknown_fraction: float = 0.12
    graph_hidden_dim: int = 128
    graph_embedding_dim: int = 64
    projection_dim: int = 32
    adapter_dim: int = 24
    graph_dropout: float = 0.12
    initial_epochs: int = 40
    imprinting_epochs: int = 6
    consolidation_epochs: int = 26
    classifier_tuning_epochs: int = 8
    base_lr: float = 1.2e-3
    adapter_lr: float = 7.5e-4
    head_lr: float = 1.5e-3
    weight_decay: float = 1e-4
    batch_size: int = 512
    replay_per_class: int = 1500
    laplace_prior_precision: float = 8.0
    posterior_draws: int = 70
    max_posterior_sd: float = 0.75
    covariance_expansion: float = 1.50
    conformal_alpha: float = 0.10
    adaptive_conformal_gamma: float = 0.005
    online_label_delay: int = 1
    max_known_rejection: float = 0.08
    prototype_tail_quantile: float = 0.975
    epistemic_quantile: float = 0.975
    controlled_drift_sequences: int = 10
    controlled_drift_batches: int = 20
    controlled_drift_position: int = 10
    device: str = 'cuda' if TORCH_AVAILABLE and torch.cuda.is_available() else 'cpu'

cfg = ProjectConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(cfg.seed)

## 2. Data discovery and integrity checks

The data layer resolves CSV files, records dataset fingerprints, removes non-predictive identifiers from the feature matrix, and creates audit files for reproducibility.

In [ ]:
def list_csv_files(data_dir: str) -> List[Path]:
    root = Path(data_dir)
    if not root.exists():
        raise FileNotFoundError(f'Dataset folder not found: {root.resolve()}')
    files = sorted(root.rglob('*.csv'))
    if not files:
        raise FileNotFoundError(f'No CSV files found under {root.resolve()}')
    return files


def file_sha256(path: Path, chunk_size: int = 1_048_576) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def build_manifest(data_dir: str, dataset_name: str) -> pd.DataFrame:
    rows = []
    for p in list_csv_files(data_dir):
        rows.append({
            'dataset': dataset_name,
            'path': str(p),
            'file_name': p.name,
            'size_bytes': p.stat().st_size,
            'sha256': file_sha256(p),
        })
    manifest = pd.DataFrame(rows)
    out = Path(cfg.output_dir) / f'{dataset_name}_manifest.csv'
    manifest.to_csv(out, index=False)
    return manifest

primary_manifest = build_manifest(cfg.primary_data_dir, cfg.primary_dataset_name)
primary_manifest.head()

In [ ]:
def read_csv_collection(files: List[Path], label_column_candidates: Tuple[str, ...] = ('label', 'Label', 'Attack', 'Class')) -> pd.DataFrame:
    frames = []
    for p in files:
        frame = pd.read_csv(p)
        frame['source_file'] = p.name
        frames.append(frame)
    df = pd.concat(frames, axis=0, ignore_index=True)
    label_cols = [c for c in label_column_candidates if c in df.columns]
    if not label_cols:
        raise ValueError(f'No label column found. Available columns include: {list(df.columns)[:20]}')
    if label_cols[0] != 'Label':
        df = df.rename(columns={label_cols[0]: 'Label'})
    return df

raw_df = read_csv_collection([Path(p) for p in primary_manifest['path']])
raw_df.shape, raw_df['Label'].nunique()

In [ ]:
TARGET_LEAKAGE_PATTERNS = (
    'label', 'class', 'category', 'attack', 'target', 'split', 'fold', 'source_file', 'source', 'file', 'id', 'timestamp', 'time', 'date'
)


def feature_control(df: pd.DataFrame, label_col: str = 'Label') -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    y = df[label_col].astype(str).copy()
    removed = []
    candidates = []
    for col in df.columns:
        low = col.lower()
        if col == label_col or any(tok in low for tok in TARGET_LEAKAGE_PATTERNS):
            removed.append({'column': col, 'reason': 'protocol or target-related field'})
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            candidates.append(col)
        else:
            removed.append({'column': col, 'reason': 'non-numeric field'})
    X = df[candidates].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    audit = pd.DataFrame(removed)
    audit.to_csv(Path(cfg.output_dir) / 'feature_control_audit.csv', index=False)
    return X, y, audit

X_raw, y_raw, feature_audit = feature_control(raw_df)
X_raw.shape, y_raw.value_counts().head()

In [ ]:
def exact_duplicate_audit(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    tmp = X.copy()
    tmp['_label'] = y.values
    feature_hash = pd.util.hash_pandas_object(X, index=False).astype(str)
    audit = pd.DataFrame({'feature_hash': feature_hash, 'label': y.values})
    grouped = audit.groupby('feature_hash')['label'].nunique().reset_index(name='unique_labels')
    conflicts = grouped[grouped['unique_labels'] > 1]
    out = pd.DataFrame({
        'rows': [len(X)],
        'unique_feature_hashes': [audit['feature_hash'].nunique()],
        'exact_duplicate_rows': [len(X) - audit['feature_hash'].nunique()],
        'conflicting_feature_hashes': [len(conflicts)],
    })
    out.to_csv(Path(cfg.output_dir) / 'duplicate_and_conflict_audit.csv', index=False)
    return out

duplicate_audit = exact_duplicate_audit(X_raw, y_raw)
duplicate_audit

## 3. Family-aware open-world protocol

The protocol separates initially known attacks, arriving classes, permanent near- and far-OOD classes, rare unknown classes, and validation-only unknown classes. The allocation below can be used directly for CICIoT2023 and can be changed for external replication.

In [ ]:
OPEN_WORLD_PROTOCOL = {
    'benign': ['Benign'],
    'initial_known': [
        'DDoS-ICMP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood',
        'MITM-ArpSpoofing', 'Mirai-greeth_flood', 'Mirai-udpplain'
    ],
    'incremental_task_1': ['DDoS-RSTFINFlood', 'DDoS-SynonymousIP_Flood'],
    'incremental_task_2': ['DDoS-ACK_Fragmentation', 'DDoS-UDP_Fragmentation'],
    'permanent_near_ood': ['DDoS-SYN_Flood', 'DNS_Spoofing', 'Mirai-greip_flood'],
    'permanent_far_ood': ['DictionaryBruteForce', 'Recon-HostDiscovery', 'VulnerabilityScan'],
    'validation_near_ood': ['DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood'],
    'validation_far_ood': ['Recon-OSScan', 'Recon-PortScan'],
    'rare_unknown': ['Backdoor_Malware', 'BrowserHijacking', 'CommandInjection', 'DDoS-HTTP_Flood', 'DDoS-SlowLoris', 'Recon-PingSweep', 'SqlInjection', 'Uploading_Attack', 'XSS'],
}

CLASS_FAMILY = {
    'Benign': 'Benign',
    'DDoS-ICMP_Flood': 'DDoS', 'DDoS-TCP_Flood': 'DDoS', 'DDoS-UDP_Flood': 'DDoS',
    'DDoS-RSTFINFlood': 'DDoS', 'DDoS-SynonymousIP_Flood': 'DDoS', 'DDoS-ACK_Fragmentation': 'DDoS',
    'DDoS-UDP_Fragmentation': 'DDoS', 'DDoS-SYN_Flood': 'DDoS', 'DDoS-ICMP_Fragmentation': 'DDoS',
    'DDoS-PSHACK_Flood': 'DDoS', 'DDoS-HTTP_Flood': 'DDoS', 'DDoS-SlowLoris': 'DDoS',
    'MITM-ArpSpoofing': 'Spoofing', 'DNS_Spoofing': 'Spoofing',
    'Mirai-greeth_flood': 'Mirai', 'Mirai-udpplain': 'Mirai', 'Mirai-greip_flood': 'Mirai',
    'DictionaryBruteForce': 'Credential',
    'Recon-HostDiscovery': 'Recon', 'Recon-OSScan': 'Recon', 'Recon-PortScan': 'Recon', 'Recon-PingSweep': 'Recon',
    'VulnerabilityScan': 'Recon',
    'Backdoor_Malware': 'Malware',
    'BrowserHijacking': 'Web', 'CommandInjection': 'Web', 'SqlInjection': 'Web', 'Uploading_Attack': 'Web', 'XSS': 'Web',
}


def assign_protocol_roles(y: pd.Series) -> pd.DataFrame:
    rows = []
    for label in y.astype(str):
        role = 'unused'
        task = '-'
        ood_type = 'known'
        for key, labels in OPEN_WORLD_PROTOCOL.items():
            if label in labels:
                role = key
                break
        if role == 'benign':
            task = '0'; ood_type = 'known'
        elif role == 'initial_known':
            task = '0'; ood_type = 'known'
        elif role == 'incremental_task_1':
            task = '1'; ood_type = 'arriving'
        elif role == 'incremental_task_2':
            task = '2'; ood_type = 'arriving'
        elif role == 'permanent_near_ood':
            ood_type = 'near'
        elif role == 'permanent_far_ood':
            ood_type = 'far'
        elif role == 'validation_near_ood':
            ood_type = 'near'
        elif role == 'validation_far_ood':
            ood_type = 'far'
        elif role == 'rare_unknown':
            ood_type = 'rare'
        rows.append({'label': label, 'protocol_role': role, 'task': task, 'family': CLASS_FAMILY.get(label, 'Other'), 'ood_type': ood_type})
    return pd.DataFrame(rows)

role_df = assign_protocol_roles(y_raw)
role_df.value_counts(['protocol_role', 'ood_type']).reset_index(name='rows')

In [ ]:
def sample_role_partition(df: pd.DataFrame, roles: pd.DataFrame, seed: int) -> pd.DataFrame:
    set_all_seeds(seed)
    meta = roles.copy()
    meta['row_index'] = np.arange(len(meta))
    meta['partition'] = 'holdout'
    initial_mask = meta['protocol_role'].isin(['benign', 'initial_known'])
    train_candidates = meta[initial_mask]
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    idx_train, idx_rest = next(splitter.split(train_candidates[['row_index']], train_candidates['label']))
    train_rows = train_candidates.iloc[idx_train]['row_index'].values
    rest_rows = train_candidates.iloc[idx_rest]['row_index'].values
    meta.loc[meta['row_index'].isin(train_rows), 'partition'] = 'train'
    rest_frame = meta.loc[meta['row_index'].isin(rest_rows)].copy()
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=0.35, random_state=seed + 1)
    cal_idx, stream_anchor_idx = next(splitter2.split(rest_frame[['row_index']], rest_frame['label']))
    cal_rows = rest_frame.iloc[cal_idx]['row_index'].values
    stream_anchor_rows = rest_frame.iloc[stream_anchor_idx]['row_index'].values
    meta.loc[meta['row_index'].isin(cal_rows), 'partition'] = 'calibration'
    stream_anchor = meta.loc[meta['row_index'].isin(stream_anchor_rows)].copy()
    splitter3 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed + 2)
    stream_idx, anchor_idx = next(splitter3.split(stream_anchor[['row_index']], stream_anchor['label']))
    meta.loc[meta['row_index'].isin(stream_anchor.iloc[stream_idx]['row_index'].values), 'partition'] = 'stream'
    meta.loc[meta['row_index'].isin(stream_anchor.iloc[anchor_idx]['row_index'].values), 'partition'] = 'anchor'
    unknown_mask = ~initial_mask
    meta.loc[unknown_mask, 'partition'] = 'open_world_pool'
    return meta

partition_df = sample_role_partition(raw_df, role_df, cfg.seed)
partition_df.value_counts(['partition', 'protocol_role']).reset_index(name='rows').head(20)

## 4. Feature preprocessing

All preprocessing parameters are fitted only on the training partition, then applied to calibration, stream, anchor, OOD, and external data roles.

In [ ]:
class FeaturePipeline:
    def __init__(self, feature_cap: int = 40, clip_quantiles: Tuple[float, float] = (0.001, 0.999)):
        self.feature_cap = feature_cap
        self.clip_quantiles = clip_quantiles
        self.columns_: Optional[List[str]] = None
        self.lower_: Optional[pd.Series] = None
        self.upper_: Optional[pd.Series] = None
        self.variance_: Optional[VarianceThreshold] = None
        self.scaler_: Optional[StandardScaler] = None
        self.selected_columns_: Optional[List[str]] = None

    def fit(self, X: pd.DataFrame) -> 'FeaturePipeline':
        X = X.copy().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        self.columns_ = X.columns.tolist()
        self.lower_ = X.quantile(self.clip_quantiles[0])
        self.upper_ = X.quantile(self.clip_quantiles[1])
        X = X.clip(self.lower_, self.upper_, axis=1)
        skew = X.skew(numeric_only=True).abs().sort_values(ascending=False)
        nonnegative = (X.min(axis=0) >= 0)
        log_cols = [c for c in skew.index if skew[c] > 2.0 and nonnegative[c]]
        X.loc[:, log_cols] = np.log1p(X[log_cols])
        self.variance_ = VarianceThreshold(threshold=1e-10)
        Xv = self.variance_.fit_transform(X)
        kept = np.array(self.columns_)[self.variance_.get_support()].tolist()
        var_scores = pd.Series(np.var(Xv, axis=0), index=kept).sort_values(ascending=False)
        self.selected_columns_ = var_scores.index[:self.feature_cap].tolist()
        self.scaler_ = StandardScaler().fit(X[self.selected_columns_])
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        if self.selected_columns_ is None or self.scaler_ is None:
            raise RuntimeError('FeaturePipeline must be fitted before transform.')
        X = X.reindex(columns=self.columns_, fill_value=0.0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        X = X.clip(self.lower_, self.upper_, axis=1)
        skew = X.skew(numeric_only=True).abs().sort_values(ascending=False)
        nonnegative = (X.min(axis=0) >= 0)
        log_cols = [c for c in skew.index if skew[c] > 2.0 and nonnegative[c] and c in self.selected_columns_]
        X.loc[:, log_cols] = np.log1p(X[log_cols])
        return self.scaler_.transform(X[self.selected_columns_]).astype(np.float32)

train_idx = partition_df.loc[partition_df['partition'] == 'train', 'row_index'].values
feature_pipe = FeaturePipeline(feature_cap=cfg.feature_cap).fit(X_raw.iloc[train_idx])
X_all = feature_pipe.transform(X_raw)
label_encoder = LabelEncoder().fit(y_raw.iloc[train_idx].astype(str))
known_labels_initial = label_encoder.classes_.tolist()
len(feature_pipe.selected_columns_), X_all.shape

## 5. Dynamic mutual k-nearest-neighbour graph construction

Each snapshot is represented as a flow-similarity graph. Nodes are flow records and edges are retained through mutual k-nearest-neighbour filtering.

In [ ]:
def build_mutual_knn_edges(X: np.ndarray, k: int = 12, metric: str = 'euclidean') -> Tuple[np.ndarray, Dict[str, float]]:
    n = X.shape[0]
    k_eff = min(k + 1, n)
    nbrs = NearestNeighbors(n_neighbors=k_eff, metric=metric, n_jobs=-1)
    nbrs.fit(X)
    distances, indices = nbrs.kneighbors(X)
    neighbor_sets = [set(row[1:]) for row in indices]
    edges = []
    knn_dists = []
    for i in range(n):
        for jpos, j in enumerate(indices[i][1:], start=1):
            if i in neighbor_sets[j]:
                edges.append((i, int(j)))
                edges.append((int(j), i))
                knn_dists.append(float(distances[i, jpos]))
    if not edges:
        edges = [(i, i) for i in range(n)]
    edge_index = np.asarray(edges, dtype=np.int64).T
    degree = np.bincount(edge_index[0], minlength=n)
    diagnostics = {
        'nodes': float(n),
        'edges': float(edge_index.shape[1]),
        'mean_degree': float(np.mean(degree)),
        'degree_cv': float(np.std(degree) / (np.mean(degree) + 1e-12)),
        'density': float(edge_index.shape[1] / max(n * (n - 1), 1)),
        'mean_knn_distance': float(np.mean(knn_dists)) if knn_dists else 0.0,
        'p90_knn_distance': float(np.quantile(knn_dists, 0.90)) if knn_dists else 0.0,
        'mutual_ratio': float(edge_index.shape[1] / max(n * k * 2, 1)),
        'isolated_node_rate': float(np.mean(degree == 0)),
    }
    return edge_index, diagnostics


def graph_homophily(edge_index: np.ndarray, y: np.ndarray) -> Dict[str, float]:
    src, dst = edge_index
    if len(src) == 0:
        return {'edge_homophily': 0.0, 'chance_label_agreement': 0.0, 'homophily_lift': 0.0}
    edge_h = float(np.mean(y[src] == y[dst]))
    counts = pd.Series(y).value_counts(normalize=True)
    chance = float(np.sum(counts.values ** 2))
    return {'edge_homophily': edge_h, 'chance_label_agreement': chance, 'homophily_lift': edge_h - chance}

In [ ]:
def make_pyg_graph(X: np.ndarray, y: np.ndarray, k: int, device: str = 'cpu'):
    edge_index, diag = build_mutual_knn_edges(X, k=k)
    diag = {**diag, **graph_homophily(edge_index, y)}
    if not TORCH_AVAILABLE or not PYG_AVAILABLE:
        return {'x': X, 'y': y, 'edge_index': edge_index, 'diagnostics': diag}
    data = Data(
        x=torch.tensor(X, dtype=torch.float32),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        y=torch.tensor(y, dtype=torch.long),
    )
    return data.to(device), diag

# Example graph audit on the first available training slice.
audit_rows = train_idx[:3500]
y_train_encoded = LabelEncoder().fit_transform(y_raw.iloc[audit_rows].astype(str))
graph_object, graph_diag = make_pyg_graph(X_all[audit_rows], y_train_encoded, cfg.knn_k, cfg.device)
pd.DataFrame([graph_diag]).to_csv(Path(cfg.output_dir) / 'initial_graph_suitability_diagnostics.csv', index=False)
graph_diag

## 6. DriftGraph-BCP model components

This section defines the GraphSAGE encoder, residual adapters, projection head, cosine classifier, and diagonal-Laplace posterior.

In [ ]:
if TORCH_AVAILABLE:
    class ResidualAdapter(nn.Module):
        def __init__(self, dim: int, bottleneck: int, dropout: float):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(dim, bottleneck),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(bottleneck, dim),
            )
        def forward(self, x):
            return x + self.net(x)

    class ContinualGraphSAGE(nn.Module):
        def __init__(self, in_dim: int, hidden_dim: int, emb_dim: int, proj_dim: int, adapter_dim: int, dropout: float):
            super().__init__()
            self.conv1 = SAGEConv(in_dim, hidden_dim) if PYG_AVAILABLE else nn.Linear(in_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, emb_dim) if PYG_AVAILABLE else nn.Linear(hidden_dim, emb_dim)
            self.norm1 = nn.LayerNorm(hidden_dim)
            self.norm2 = nn.LayerNorm(emb_dim)
            self.adapter1 = ResidualAdapter(hidden_dim, adapter_dim, dropout)
            self.adapter2 = ResidualAdapter(emb_dim, adapter_dim, dropout)
            self.dropout = nn.Dropout(dropout)
            self.proj = nn.Sequential(nn.Linear(emb_dim, proj_dim), nn.GELU(), nn.Linear(proj_dim, proj_dim))

        def forward(self, x, edge_index=None):
            if PYG_AVAILABLE and edge_index is not None:
                h = self.conv1(x, edge_index)
                h = self.adapter1(self.norm1(F.gelu(h)))
                h = self.dropout(h)
                z = self.conv2(h, edge_index)
            else:
                h = self.conv1(x)
                h = self.adapter1(self.norm1(F.gelu(h)))
                h = self.dropout(h)
                z = self.conv2(h)
            z = self.adapter2(self.norm2(F.gelu(z)))
            p = F.normalize(self.proj(z), dim=-1)
            return z, p

    class CosineClassifier(nn.Module):
        def __init__(self, emb_dim: int, n_classes: int, scale: float = 16.0):
            super().__init__()
            self.weight = nn.Parameter(torch.randn(n_classes, emb_dim))
            nn.init.xavier_uniform_(self.weight)
            self.scale = nn.Parameter(torch.tensor(float(scale)))

        def forward(self, z):
            z_norm = F.normalize(z, dim=-1)
            w_norm = F.normalize(self.weight, dim=-1)
            return self.scale.clamp(1.0, 64.0) * z_norm @ w_norm.T
else:
    ResidualAdapter = None
    ContinualGraphSAGE = None
    CosineClassifier = None

In [ ]:
class ReplayBuffer:
    def __init__(self, per_class: int, seed: int):
        self.per_class = per_class
        self.seed = seed
        self.storage: Dict[int, List[int]] = {}

    def add_examples(self, indices: np.ndarray, labels: np.ndarray, uncertainty: Optional[np.ndarray] = None) -> None:
        rng = np.random.default_rng(self.seed)
        for cls in np.unique(labels):
            cls_idx = indices[labels == cls]
            if uncertainty is not None:
                cls_unc = uncertainty[labels == cls]
                order = np.argsort(cls_unc)[::-1]
                selected = cls_idx[order[:self.per_class]]
            else:
                selected = rng.choice(cls_idx, size=min(self.per_class, len(cls_idx)), replace=False)
            current = self.storage.get(int(cls), [])
            combined = np.unique(np.concatenate([np.asarray(current, dtype=int), selected])).astype(int)
            if len(combined) > self.per_class:
                combined = rng.choice(combined, size=self.per_class, replace=False)
            self.storage[int(cls)] = combined.tolist()

    def get_indices(self) -> np.ndarray:
        if not self.storage:
            return np.array([], dtype=int)
        return np.concatenate([np.asarray(v, dtype=int) for v in self.storage.values()])

In [ ]:
if TORCH_AVAILABLE:
    class DiagonalLaplaceHead:
        def __init__(self, prior_precision: float = 8.0, max_sd: float = 0.75):
            self.prior_precision = prior_precision
            self.max_sd = max_sd
            self.mean_: Optional[torch.Tensor] = None
            self.precision_: Optional[torch.Tensor] = None

        def fit(self, classifier: CosineClassifier, embeddings: torch.Tensor, labels: torch.Tensor) -> 'DiagonalLaplaceHead':
            classifier.eval()
            with torch.no_grad():
                logits = classifier(embeddings)
                probs = torch.softmax(logits, dim=-1)
                n_classes = probs.shape[1]
                z2 = F.normalize(embeddings, dim=-1).pow(2)
                precision = torch.full_like(classifier.weight, float(self.prior_precision))
                for c in range(n_classes):
                    pc = probs[:, c] * (1.0 - probs[:, c])
                    curvature = (pc[:, None] * z2).sum(dim=0)
                    precision[c] += curvature
                self.mean_ = classifier.weight.detach().clone()
                self.precision_ = precision.detach().clamp(min=1e-6)
            return self

        def expand_covariance(self, factor: float) -> None:
            if self.precision_ is not None:
                self.precision_ = self.precision_ / float(factor)

        def draw_weights(self, n_draws: int) -> torch.Tensor:
            if self.mean_ is None or self.precision_ is None:
                raise RuntimeError('Laplace head must be fitted before drawing weights.')
            sd = torch.rsqrt(self.precision_).clamp(max=self.max_sd)
            eps = torch.randn((n_draws,) + self.mean_.shape, device=self.mean_.device)
            return self.mean_.unsqueeze(0) + eps * sd.unsqueeze(0)

        def posterior_predict(self, embeddings: torch.Tensor, scale: torch.Tensor, n_draws: int) -> Dict[str, torch.Tensor]:
            weights = F.normalize(self.draw_weights(n_draws), dim=-1)
            z = F.normalize(embeddings, dim=-1)
            logits = scale.clamp(1.0, 64.0) * torch.einsum('nd,scd->snc', z, weights)
            probs = torch.softmax(logits, dim=-1)
            mean_probs = probs.mean(dim=0)
            predictive_entropy = -(mean_probs * mean_probs.clamp_min(1e-12).log()).sum(dim=-1)
            expected_entropy = -(probs * probs.clamp_min(1e-12).log()).sum(dim=-1).mean(dim=0)
            mutual_information = predictive_entropy - expected_entropy
            prob_variance = probs.var(dim=0).mean(dim=-1)
            return {
                'probs': mean_probs,
                'predictive_entropy': predictive_entropy,
                'mutual_information': mutual_information,
                'probability_variance': prob_variance,
                'draws': probs,
            }
else:
    DiagonalLaplaceHead = None

## 7. Training objectives

The continual objective combines classification, supervised contrastive learning, logit distillation, feature distillation, prototype anchoring, and online EWC terms.

In [ ]:
if TORCH_AVAILABLE:
    def supervised_contrastive_loss(features: torch.Tensor, labels: torch.Tensor, temperature: float = 0.20) -> torch.Tensor:
        features = F.normalize(features, dim=-1)
        sim = features @ features.T / temperature
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(features.device)
        logits_mask = torch.ones_like(mask) - torch.eye(mask.shape[0], device=features.device)
        mask = mask * logits_mask
        exp_sim = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True).clamp_min(1e-12))
        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        return -mean_log_prob_pos.mean()

    def distillation_loss(student_logits, teacher_logits, temperature: float = 2.0) -> torch.Tensor:
        t = float(temperature)
        p_teacher = F.softmax(teacher_logits / t, dim=-1)
        log_student = F.log_softmax(student_logits / t, dim=-1)
        return F.kl_div(log_student, p_teacher, reduction='batchmean') * (t ** 2)

    def feature_distillation_loss(student_z, teacher_z) -> torch.Tensor:
        return 1.0 - F.cosine_similarity(student_z, teacher_z, dim=-1).mean()

    def prototype_anchor_loss(embeddings: torch.Tensor, labels: torch.Tensor, anchors: Dict[int, torch.Tensor]) -> torch.Tensor:
        if not anchors:
            return torch.tensor(0.0, device=embeddings.device)
        losses = []
        for cls, proto in anchors.items():
            mask = labels == int(cls)
            if mask.any():
                losses.append(1.0 - F.cosine_similarity(embeddings[mask], proto.to(embeddings.device).view(1, -1), dim=-1).mean())
        return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=embeddings.device)

    class OnlineEWC:
        def __init__(self, decay: float = 0.90):
            self.decay = decay
            self.theta_ref: Dict[str, torch.Tensor] = {}
            self.fisher: Dict[str, torch.Tensor] = {}

        def consolidate(self, model: nn.Module, dataloader: DataLoader, loss_fn) -> None:
            model.eval()
            fisher_new = {n: torch.zeros_like(p, device=p.device) for n, p in model.named_parameters() if p.requires_grad}
            for batch in dataloader:
                model.zero_grad(set_to_none=True)
                loss = loss_fn(batch)
                loss.backward()
                for n, p in model.named_parameters():
                    if p.requires_grad and p.grad is not None:
                        fisher_new[n] += p.grad.detach().pow(2)
            for n in fisher_new:
                fisher_new[n] /= max(len(dataloader), 1)
                if n in self.fisher:
                    self.fisher[n] = self.decay * self.fisher[n] + (1 - self.decay) * fisher_new[n]
                else:
                    self.fisher[n] = fisher_new[n]
                self.theta_ref[n] = dict(model.named_parameters())[n].detach().clone()

        def penalty(self, model: nn.Module) -> torch.Tensor:
            if not self.theta_ref:
                return torch.tensor(0.0, device=next(model.parameters()).device)
            total = torch.tensor(0.0, device=next(model.parameters()).device)
            for n, p in model.named_parameters():
                if n in self.theta_ref:
                    total = total + (self.fisher[n] * (p - self.theta_ref[n]).pow(2)).sum()
            return total
else:
    OnlineEWC = None

In [ ]:
@dataclass
class ObjectiveWeights:
    supervised_contrastive: float = 0.15
    prototype_geometry: float = 0.08
    logit_distillation: float = 0.80
    feature_distillation: float = 0.25
    prototype_anchor: float = 0.35
    ewc: float = 0.10

objective_weights = ObjectiveWeights()

## 8. Calibration and conformal prediction

Temperature and posterior-scale selection calibrate posterior-predictive probabilities. Hierarchical class-conditional RAPS returns prediction sets with a target miscoverage level.

In [ ]:
def expected_calibration_error(conf: np.ndarray, correct: np.ndarray, n_bins: int = 12) -> Tuple[float, pd.DataFrame]:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    n = len(conf)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        count = int(mask.sum())
        if count == 0:
            continue
        acc = float(correct[mask].mean())
        cmean = float(conf[mask].mean())
        gap = acc - cmean
        ece += (count / n) * abs(gap)
        rows.append({'confidence_lower': lo, 'confidence_upper': hi, 'mean_confidence': cmean, 'empirical_accuracy': acc, 'count': count, 'accuracy_minus_confidence': gap})
    return float(ece), pd.DataFrame(rows)


def multiclass_brier_score(y_true: np.ndarray, probs: np.ndarray, n_classes: int) -> float:
    y_onehot = np.eye(n_classes)[y_true]
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))


def tune_temperature(logits: np.ndarray, y: np.ndarray, grid: Iterable[float] = np.linspace(0.5, 3.0, 26)) -> float:
    best_t, best_loss = 1.0, np.inf
    for t in grid:
        probs = np.exp(logits / t)
        probs = probs / probs.sum(axis=1, keepdims=True)
        loss = log_loss(y, probs, labels=np.arange(probs.shape[1]))
        if loss < best_loss:
            best_t, best_loss = float(t), float(loss)
    return best_t

class RAPSCalibrator:
    def __init__(self, alpha: float = 0.10, k_reg: int = 5, lam: float = 0.01):
        self.alpha = alpha
        self.k_reg = k_reg
        self.lam = lam
        self.global_q_: float = 1.0
        self.class_q_: Dict[int, float] = {}

    def nonconformity(self, probs: np.ndarray, y: np.ndarray) -> np.ndarray:
        order = np.argsort(-probs, axis=1)
        sorted_probs = np.take_along_axis(probs, order, axis=1)
        cum = np.cumsum(sorted_probs, axis=1)
        ranks = np.zeros(len(y), dtype=int)
        scores = np.zeros(len(y), dtype=float)
        for i in range(len(y)):
            rank = int(np.where(order[i] == y[i])[0][0])
            ranks[i] = rank
            penalty = self.lam * max(rank + 1 - self.k_reg, 0)
            scores[i] = cum[i, rank] + penalty
        return scores

    def fit(self, probs: np.ndarray, y: np.ndarray) -> 'RAPSCalibrator':
        scores = self.nonconformity(probs, y)
        n = len(scores)
        q_level = min(1.0, math.ceil((n + 1) * (1 - self.alpha)) / max(n, 1))
        self.global_q_ = float(np.quantile(scores, q_level, method='higher'))
        self.class_q_.clear()
        for cls in np.unique(y):
            cls_scores = scores[y == cls]
            if len(cls_scores) >= 30:
                q_level_c = min(1.0, math.ceil((len(cls_scores) + 1) * (1 - self.alpha)) / max(len(cls_scores), 1))
                self.class_q_[int(cls)] = float(np.quantile(cls_scores, q_level_c, method='higher'))
        return self

    def predict_sets(self, probs: np.ndarray) -> List[List[int]]:
        sets = []
        order = np.argsort(-probs, axis=1)
        sorted_probs = np.take_along_axis(probs, order, axis=1)
        cum = np.cumsum(sorted_probs, axis=1)
        for i in range(probs.shape[0]):
            included = []
            for rank, cls in enumerate(order[i]):
                penalty = self.lam * max(rank + 1 - self.k_reg, 0)
                score = cum[i, rank] + penalty
                threshold = self.class_q_.get(int(cls), self.global_q_)
                if score <= threshold:
                    included.append(int(cls))
            sets.append(included)
        return sets

## 9. Prototype, energy, and open-world evidence fusion

The unknown detector fuses posterior confidence, uncertainty, conformal behaviour, energy, and prototype geometry while constraining known-class rejection.

In [ ]:
def compute_prototypes(embeddings: np.ndarray, labels: np.ndarray) -> Dict[int, np.ndarray]:
    return {int(c): embeddings[labels == c].mean(axis=0) for c in np.unique(labels)}


def prototype_features(embeddings: np.ndarray, prototypes: Dict[int, np.ndarray]) -> pd.DataFrame:
    classes = sorted(prototypes.keys())
    proto_mat = np.vstack([prototypes[c] for c in classes])
    dists = np.linalg.norm(embeddings[:, None, :] - proto_mat[None, :, :], axis=2)
    cosine = 1.0 - (embeddings @ proto_mat.T) / ((np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12) * (np.linalg.norm(proto_mat, axis=1)[None, :] + 1e-12))
    part = np.partition(dists, kth=min(1, dists.shape[1]-1), axis=1)
    margin = part[:, 1] - part[:, 0] if dists.shape[1] > 1 else np.ones(len(embeddings))
    return pd.DataFrame({
        'min_euclidean_prototype_distance': dists.min(axis=1),
        'min_cosine_prototype_distance': cosine.min(axis=1),
        'prototype_margin': margin,
    })


def energy_score(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = logits / temperature
    max_z = z.max(axis=1, keepdims=True)
    return -temperature * (max_z.squeeze() + np.log(np.exp(z - max_z).sum(axis=1)))


def open_world_feature_matrix(probs: np.ndarray, logits: np.ndarray, entropy: np.ndarray, mi: np.ndarray, var: np.ndarray, prediction_sets: List[List[int]], proto_df: pd.DataFrame) -> pd.DataFrame:
    sorted_probs = -np.sort(-probs, axis=1)
    max_prob = sorted_probs[:, 0]
    margin = sorted_probs[:, 0] - sorted_probs[:, 1] if probs.shape[1] > 1 else sorted_probs[:, 0]
    set_size = np.asarray([len(s) for s in prediction_sets])
    df = pd.DataFrame({
        'one_minus_max_probability': 1 - max_prob,
        'one_minus_margin': 1 - margin,
        'entropy': entropy,
        'mutual_information': mi,
        'posterior_variance': var,
        'energy': energy_score(logits),
        'conformal_set_size': set_size,
        'non_singleton': (set_size != 1).astype(float),
    })
    return pd.concat([df.reset_index(drop=True), proto_df.reset_index(drop=True)], axis=1)

class OpenWorldFusion:
    def __init__(self, max_known_rejection: float = 0.08):
        self.max_known_rejection = max_known_rejection
        self.model = LogisticRegression(max_iter=2000, class_weight='balanced')
        self.threshold_: float = 0.5
        self.feature_columns_: Optional[List[str]] = None

    def fit(self, X_open: pd.DataFrame, is_unknown: np.ndarray) -> 'OpenWorldFusion':
        self.feature_columns_ = X_open.columns.tolist()
        self.model.fit(X_open[self.feature_columns_], is_unknown)
        scores = self.model.predict_proba(X_open[self.feature_columns_])[:, 1]
        known_scores = scores[is_unknown == 0]
        self.threshold_ = float(np.quantile(known_scores, 1 - self.max_known_rejection))
        return self

    def score(self, X_open: pd.DataFrame) -> np.ndarray:
        if self.feature_columns_ is None:
            raise RuntimeError('OpenWorldFusion must be fitted before scoring.')
        return self.model.predict_proba(X_open[self.feature_columns_])[:, 1]

    def decide_unknown(self, X_open: pd.DataFrame) -> np.ndarray:
        return (self.score(X_open) >= self.threshold_).astype(int)

## 10. Drift monitoring

The monitoring layer converts predictive, representation, graph-structural, prototype, and class-prevalence evidence into calibrated anomaly percentiles. BOCPD and a delayed supervised error monitor then produce operational drift events.

In [ ]:
class PercentileReference:
    def __init__(self):
        self.reference_: Optional[np.ndarray] = None

    def fit(self, values: np.ndarray) -> 'PercentileReference':
        self.reference_ = np.asarray(values, dtype=float)
        return self

    def transform(self, values: np.ndarray) -> np.ndarray:
        if self.reference_ is None:
            raise RuntimeError('Reference distribution is not fitted.')
        ref = np.sort(self.reference_)
        ranks = np.searchsorted(ref, values, side='right')
        return ranks / max(len(ref), 1)

class StudentTBOCPD:
    def __init__(self, hazard: float = 0.05, max_run_length: int = 200):
        self.hazard = hazard
        self.max_run_length = max_run_length
        self.reset()

    def reset(self):
        self.run_probs = np.array([1.0])
        self.values: List[float] = []

    def step(self, value: float) -> Dict[str, float]:
        self.values.append(float(value))
        r = len(self.run_probs)
        predictive = np.ones(r)
        for run in range(r):
            window = np.asarray(self.values[-max(run, 1):])
            loc = window.mean() if len(window) else 0.0
            scale = window.std(ddof=1) + 1e-3 if len(window) > 1 else 1.0
            predictive[run] = stats.t.pdf(value, df=max(len(window)-1, 1), loc=loc, scale=scale) + 1e-12
        growth = self.run_probs * (1 - self.hazard) * predictive
        change = np.sum(self.run_probs * self.hazard * predictive)
        new_probs = np.concatenate([[change], growth])
        new_probs = new_probs[:self.max_run_length]
        new_probs = new_probs / new_probs.sum()
        self.run_probs = new_probs
        short_mass = float(new_probs[:3].sum())
        expected_run = float(np.sum(np.arange(len(new_probs)) * new_probs))
        return {'short_run_mass': short_mass, 'expected_run_length': expected_run, 'surprise': float(-np.log(np.max(predictive)))}

class DelayedErrorMonitor:
    def __init__(self, baseline_error: float = 0.08, margin: float = 0.05, a0: float = 2.0, b0: float = 20.0):
        self.baseline_error = baseline_error
        self.margin = margin
        self.a0 = a0
        self.b0 = b0

    def probability_of_concept_shift(self, errors: np.ndarray) -> float:
        e = int(errors.sum())
        n = int(len(errors))
        threshold = self.baseline_error + self.margin
        return float(1 - stats.beta.cdf(threshold, self.a0 + e, self.b0 + n - e))

## 11. Prequential stream construction

Each snapshot is predicted before the delayed labels for that snapshot are used by the adaptation policy. Arriving classes are evaluated as unknown during pre-adaptation phases and as known after class assimilation.

In [ ]:
def draw_stratified_rows(meta: pd.DataFrame, labels: List[str], size: int, rng: np.random.Generator) -> np.ndarray:
    pool = meta[meta['label'].isin(labels)]['row_index'].values
    if len(pool) == 0:
        return np.array([], dtype=int)
    replace = len(pool) < size
    return rng.choice(pool, size=size, replace=replace)


def build_stream_schedule(meta: pd.DataFrame, config: ProjectConfig, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    snapshot_id = 0
    phase_specs = [
        (0, 'initial_post', OPEN_WORLD_PROTOCOL['initial_known'], []),
        (1, 'pre', OPEN_WORLD_PROTOCOL['initial_known'], OPEN_WORLD_PROTOCOL['incremental_task_1']),
        (1, 'post', OPEN_WORLD_PROTOCOL['initial_known'] + OPEN_WORLD_PROTOCOL['incremental_task_1'], []),
        (2, 'pre', OPEN_WORLD_PROTOCOL['initial_known'] + OPEN_WORLD_PROTOCOL['incremental_task_1'], OPEN_WORLD_PROTOCOL['incremental_task_2']),
        (2, 'post', OPEN_WORLD_PROTOCOL['initial_known'] + OPEN_WORLD_PROTOCOL['incremental_task_1'] + OPEN_WORLD_PROTOCOL['incremental_task_2'], []),
    ]
    for task, phase, known_attacks, arriving in phase_specs:
        for _ in range(config.snapshots_per_phase):
            n = config.stream_snapshot_size
            n_benign = int(n * config.benign_stream_fraction)
            n_arriving = int(n * config.arriving_unknown_fraction) if phase == 'pre' else 0
            n_perm = int(n * config.permanent_unknown_fraction)
            n_known_attack = n - n_benign - n_arriving - n_perm
            idx = []
            idx.extend(draw_stratified_rows(meta, OPEN_WORLD_PROTOCOL['benign'], n_benign, rng))
            idx.extend(draw_stratified_rows(meta, known_attacks, n_known_attack, rng))
            if n_arriving > 0:
                idx.extend(draw_stratified_rows(meta, arriving, n_arriving, rng))
            permanent = OPEN_WORLD_PROTOCOL['permanent_near_ood'] + OPEN_WORLD_PROTOCOL['permanent_far_ood']
            idx.extend(draw_stratified_rows(meta, permanent, n_perm, rng))
            rng.shuffle(idx)
            for row_index in idx:
                role = meta.loc[meta['row_index'] == row_index].iloc[0]
                rows.append({'snapshot': snapshot_id, 'task': task, 'phase': phase, 'row_index': int(row_index), 'label': role['label'], 'protocol_role': role['protocol_role'], 'ood_type': role['ood_type']})
            snapshot_id += 1
    return pd.DataFrame(rows)

stream_df = build_stream_schedule(partition_df, cfg, cfg.seed)
stream_df.groupby(['task', 'phase'])['snapshot'].nunique().reset_index(name='snapshots')

## 12. Model fitting and snapshot evaluation functions

The functions below implement initial fitting, class expansion, posterior refresh, snapshot inference, and metric computation. They are designed so that the proposed model, baselines, and ablations share the same data protocol.

In [ ]:
def classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: Optional[np.ndarray] = None) -> Dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
        'mcc': float(matthews_corrcoef(y_true, y_pred)),
    }
    if probs is not None and probs.shape[1] > 1:
        try:
            out['macro_auroc'] = float(roc_auc_score(y_true, probs, multi_class='ovr', average='macro'))
        except Exception:
            out['macro_auroc'] = np.nan
        try:
            y_onehot = np.eye(probs.shape[1])[y_true]
            out['macro_auprc'] = float(np.mean([average_precision_score(y_onehot[:, c], probs[:, c]) for c in range(probs.shape[1])]))
        except Exception:
            out['macro_auprc'] = np.nan
    return out


def open_world_metrics(is_unknown: np.ndarray, unknown_score: np.ndarray, unknown_pred: np.ndarray) -> Dict[str, float]:
    prec, rec, f1, _ = precision_recall_fscore_support(is_unknown, unknown_pred, average='binary', zero_division=0)
    tn, fp, fn, tp = confusion_matrix(is_unknown, unknown_pred, labels=[0,1]).ravel()
    out = {
        'unknown_precision': float(prec),
        'unknown_recall': float(rec),
        'unknown_f1': float(f1),
        'false_known_rate': float(fn / max(fn + tp, 1)),
        'false_unknown_rate': float(fp / max(fp + tn, 1)),
        'balanced_accuracy': float(balanced_accuracy_score(is_unknown, unknown_pred)),
        'mcc': float(matthews_corrcoef(is_unknown, unknown_pred)),
    }
    try:
        out['unknown_auroc'] = float(roc_auc_score(is_unknown, unknown_score))
        out['unknown_auprc'] = float(average_precision_score(is_unknown, unknown_score))
    except Exception:
        out['unknown_auroc'] = np.nan
        out['unknown_auprc'] = np.nan
    return out

In [ ]:
class DriftGraphBCPRunner:
    def __init__(self, config: ProjectConfig, variant: str = 'DriftGraph-BCP'):
        self.cfg = config
        self.variant = variant
        self.encoder = None
        self.classifier = None
        self.laplace = None
        self.raps = RAPSCalibrator(alpha=config.conformal_alpha)
        self.open_fusion = OpenWorldFusion(max_known_rejection=config.max_known_rejection)
        self.replay = ReplayBuffer(per_class=config.replay_per_class, seed=config.seed)
        self.known_classes: List[str] = []
        self.label_to_id: Dict[str, int] = {}
        self.prototypes: Dict[int, np.ndarray] = {}

    def initialize_known_classes(self) -> None:
        self.known_classes = OPEN_WORLD_PROTOCOL['benign'] + OPEN_WORLD_PROTOCOL['initial_known']
        self.label_to_id = {lab: i for i, lab in enumerate(self.known_classes)}

    def encode_known_labels(self, labels: Iterable[str]) -> np.ndarray:
        return np.asarray([self.label_to_id.get(str(x), -1) for x in labels], dtype=int)

    def fit_initial(self, X: np.ndarray, labels: pd.Series, row_indices: np.ndarray) -> None:
        self.initialize_known_classes()
        if not TORCH_AVAILABLE:
            return
        y = self.encode_known_labels(labels.iloc[row_indices])
        mask = y >= 0
        X_fit = X[row_indices][mask]
        y_fit = y[mask]
        data_obj, _ = make_pyg_graph(X_fit, y_fit, self.cfg.knn_k, self.cfg.device)
        self.encoder = ContinualGraphSAGE(X_fit.shape[1], self.cfg.graph_hidden_dim, self.cfg.graph_embedding_dim, self.cfg.projection_dim, self.cfg.adapter_dim, self.cfg.graph_dropout).to(self.cfg.device)
        self.classifier = CosineClassifier(self.cfg.graph_embedding_dim, len(self.known_classes)).to(self.cfg.device)
        params = list(self.encoder.parameters()) + list(self.classifier.parameters())
        optimizer = torch.optim.AdamW(params, lr=self.cfg.base_lr, weight_decay=self.cfg.weight_decay)
        class_weights = compute_class_weight('balanced', classes=np.unique(y_fit), y=y_fit)
        class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=self.cfg.device)
        for epoch in range(self.cfg.initial_epochs):
            self.encoder.train(); self.classifier.train()
            optimizer.zero_grad(set_to_none=True)
            z, p = self.encoder(data_obj.x, data_obj.edge_index if PYG_AVAILABLE else None)
            logits = self.classifier(z)
            loss = F.cross_entropy(logits, data_obj.y, weight=class_weights_t)
            loss = loss + objective_weights.supervised_contrastive * supervised_contrastive_loss(p, data_obj.y)
            loss.backward()
            optimizer.step()
        self.refresh_posterior(X_fit, y_fit)
        self.replay.add_examples(row_indices[mask], y_fit)

    def refresh_posterior(self, X_fit: np.ndarray, y_fit: np.ndarray) -> None:
        if not TORCH_AVAILABLE or self.encoder is None or self.classifier is None:
            return
        data_obj, _ = make_pyg_graph(X_fit, y_fit, self.cfg.knn_k, self.cfg.device)
        self.encoder.eval(); self.classifier.eval()
        with torch.no_grad():
            z, _ = self.encoder(data_obj.x, data_obj.edge_index if PYG_AVAILABLE else None)
        self.laplace = DiagonalLaplaceHead(self.cfg.laplace_prior_precision, self.cfg.max_posterior_sd).fit(self.classifier, z, data_obj.y)
        self.prototypes = compute_prototypes(z.detach().cpu().numpy(), y_fit)

    def expand_classes(self, new_classes: List[str]) -> None:
        for lab in new_classes:
            if lab not in self.known_classes:
                self.known_classes.append(lab)
        self.label_to_id = {lab: i for i, lab in enumerate(self.known_classes)}
        if TORCH_AVAILABLE and self.classifier is not None:
            old_weight = self.classifier.weight.detach().clone()
            new_head = CosineClassifier(self.cfg.graph_embedding_dim, len(self.known_classes)).to(self.cfg.device)
            with torch.no_grad():
                new_head.weight[:old_weight.shape[0]] = old_weight
            self.classifier = new_head

    def adapt_to_task(self, X: np.ndarray, labels: pd.Series, row_indices: np.ndarray, new_classes: List[str]) -> None:
        self.expand_classes(new_classes)
        if not TORCH_AVAILABLE:
            return
        replay_idx = self.replay.get_indices()
        train_rows = np.unique(np.concatenate([row_indices, replay_idx])).astype(int)
        y = self.encode_known_labels(labels.iloc[train_rows])
        mask = y >= 0
        X_fit = X[train_rows][mask]
        y_fit = y[mask]
        data_obj, _ = make_pyg_graph(X_fit, y_fit, self.cfg.knn_k, self.cfg.device)
        optimizer = torch.optim.AdamW(list(self.encoder.parameters()) + list(self.classifier.parameters()), lr=self.cfg.adapter_lr, weight_decay=self.cfg.weight_decay)
        for epoch in range(self.cfg.consolidation_epochs):
            self.encoder.train(); self.classifier.train()
            optimizer.zero_grad(set_to_none=True)
            z, p = self.encoder(data_obj.x, data_obj.edge_index if PYG_AVAILABLE else None)
            logits = self.classifier(z)
            loss = F.cross_entropy(logits, data_obj.y)
            loss = loss + objective_weights.supervised_contrastive * supervised_contrastive_loss(p, data_obj.y)
            loss = loss + objective_weights.prototype_anchor * prototype_anchor_loss(z, data_obj.y, {k: torch.tensor(v, dtype=torch.float32) for k, v in self.prototypes.items()})
            loss.backward(); optimizer.step()
        self.refresh_posterior(X_fit, y_fit)
        self.replay.add_examples(train_rows[mask], y_fit)

    def predict_snapshot(self, X_snapshot: np.ndarray) -> Dict[str, np.ndarray]:
        if not TORCH_AVAILABLE or self.encoder is None or self.classifier is None or self.laplace is None:
            raise RuntimeError('Torch and PyG are required for model inference.')
        dummy_y = np.zeros(X_snapshot.shape[0], dtype=int)
        data_obj, _ = make_pyg_graph(X_snapshot, dummy_y, self.cfg.knn_k, self.cfg.device)
        self.encoder.eval(); self.classifier.eval()
        with torch.no_grad():
            z, _ = self.encoder(data_obj.x, data_obj.edge_index if PYG_AVAILABLE else None)
            logits = self.classifier(z)
            posterior = self.laplace.posterior_predict(z, self.classifier.scale, self.cfg.posterior_draws)
        probs = posterior['probs'].detach().cpu().numpy()
        pred = probs.argmax(axis=1)
        entropy = posterior['predictive_entropy'].detach().cpu().numpy()
        mi = posterior['mutual_information'].detach().cpu().numpy()
        var = posterior['probability_variance'].detach().cpu().numpy()
        return {'probs': probs, 'pred': pred, 'entropy': entropy, 'mutual_information': mi, 'posterior_variance': var, 'embeddings': z.detach().cpu().numpy(), 'logits': logits.detach().cpu().numpy()}

## 13. Baselines and ablation registry

Every comparison uses the same split, feature matrix, graph construction, stream, class-arrival schedule, and metric functions.

In [ ]:
BASELINE_REGISTRY = {
    'Static-GraphSAGE': {'continual': False, 'laplace': False, 'dropout_inference': False, 'replay': False, 'drift': False, 'conformal': False},
    'Replay-Continual-GraphSAGE': {'continual': True, 'laplace': False, 'dropout_inference': False, 'replay': True, 'drift': False, 'conformal': False},
    'DER-Continual-GraphSAGE': {'continual': True, 'laplace': False, 'dropout_inference': False, 'replay': True, 'logit_distillation': True, 'drift': False, 'conformal': False},
    'MC-Dropout-Continual-GraphSAGE': {'continual': True, 'laplace': False, 'dropout_inference': True, 'replay': True, 'drift': False, 'conformal': False},
    'Laplace-Continual-GraphSAGE': {'continual': True, 'laplace': True, 'dropout_inference': False, 'replay': True, 'drift': False, 'conformal': False},
}

ABLATION_REGISTRY = {
    'No-Distribution-BOCPD': {'disable_distribution_bocpd': True},
    'No-Delayed-Concept-Monitor': {'disable_delayed_monitor': True},
    'No-Replay': {'disable_replay': True},
    'No-Logit-Distillation': {'disable_logit_distillation': True},
    'No-Feature-Distillation': {'disable_feature_distillation': True},
    'No-Prototype-Anchor': {'disable_prototype_anchor': True},
    'No-EWC': {'disable_ewc': True},
    'No-SupCon': {'disable_supervised_contrastive': True},
    'No-Prototype-OOD': {'disable_prototype_ood': True},
    'APS-Instead-of-RAPS': {'use_aps': True},
    'Static-Conformal': {'disable_adaptive_conformal': True},
    'No-Covariance-Expansion': {'disable_covariance_expansion': True},
    'Full-Encoder-Adaptation': {'full_encoder_adaptation': True},
}

## 14. Experiment execution

Run the proposed model, baselines, ablations, robustness seeds, and external replication from these cells. The output artifacts are saved as CSV files and figures under the configured output folder.

In [ ]:
def run_single_seed_experiment(seed: int, variant: str = 'DriftGraph-BCP') -> Dict[str, pd.DataFrame]:
    set_all_seeds(seed)
    local_cfg = ProjectConfig(**{**asdict(cfg), 'seed': seed})
    runner = DriftGraphBCPRunner(local_cfg, variant=variant)
    train_rows = partition_df.loc[partition_df['partition'] == 'train', 'row_index'].values
    runner.fit_initial(X_all, y_raw, train_rows)
    stream = build_stream_schedule(partition_df, local_cfg, seed)
    snapshot_rows = []
    open_rows = []
    calibration_rows = []
    known_class_rows = []
    for snapshot, snap_df in stream.groupby('snapshot'):
        phase = snap_df['phase'].iloc[0]
        task = int(snap_df['task'].iloc[0])
        idx = snap_df['row_index'].values
        preds = runner.predict_snapshot(X_all[idx])
        known_y = runner.encode_known_labels(y_raw.iloc[idx])
        is_known = known_y >= 0
        is_unknown = (~is_known).astype(int)
        pred_ids = preds['pred']
        probs = preds['probs']
        pred_known = pred_ids[is_known]
        true_known = known_y[is_known]
        if is_known.any():
            cm = classification_metrics(true_known, pred_known, probs[is_known])
            cm.update({'snapshot': snapshot, 'task': task, 'phase': phase})
            snapshot_rows.append(cm)
        raps_sets = [[int(np.argmax(p))] for p in probs]
        proto_df = prototype_features(preds['embeddings'], runner.prototypes) if runner.prototypes else pd.DataFrame(index=np.arange(len(idx)))
        open_X = open_world_feature_matrix(probs, preds['logits'], preds['entropy'], preds['mutual_information'], preds['posterior_variance'], raps_sets, proto_df)
        if hasattr(runner.open_fusion, 'feature_columns_') and runner.open_fusion.feature_columns_ is not None:
            unknown_score = runner.open_fusion.score(open_X)
            unknown_pred = (unknown_score >= runner.open_fusion.threshold_).astype(int)
        else:
            unknown_score = open_X['one_minus_max_probability'].values
            threshold = np.quantile(unknown_score[is_unknown == 0], 1 - local_cfg.max_known_rejection) if np.any(is_unknown == 0) else 0.5
            unknown_pred = (unknown_score >= threshold).astype(int)
        om = open_world_metrics(is_unknown, unknown_score, unknown_pred)
        om.update({'snapshot': snapshot, 'task': task, 'phase': phase})
        open_rows.append(om)
        conf = probs.max(axis=1)
        correct = (pred_ids == np.clip(known_y, 0, len(runner.known_classes)-1)) & is_known
        ece, bin_df = expected_calibration_error(conf[is_known], correct[is_known].astype(int)) if is_known.any() else (np.nan, pd.DataFrame())
        calibration_rows.append({'snapshot': snapshot, 'task': task, 'phase': phase, 'ece': ece, 'mean_entropy': float(np.mean(preds['entropy'])), 'mean_mi': float(np.mean(preds['mutual_information'])), 'mean_variance': float(np.mean(preds['posterior_variance']))})
        if phase == 'post' and task in (1, 2):
            pass
        if phase == 'pre' and task == 1 and snapshot % local_cfg.snapshots_per_phase == local_cfg.snapshots_per_phase - 1:
            new_rows = stream.loc[(stream['task'] == task) & (stream['phase'] == phase), 'row_index'].values
            runner.adapt_to_task(X_all, y_raw, new_rows, OPEN_WORLD_PROTOCOL['incremental_task_1'])
        if phase == 'pre' and task == 2 and snapshot % local_cfg.snapshots_per_phase == local_cfg.snapshots_per_phase - 1:
            new_rows = stream.loc[(stream['task'] == task) & (stream['phase'] == phase), 'row_index'].values
            runner.adapt_to_task(X_all, y_raw, new_rows, OPEN_WORLD_PROTOCOL['incremental_task_2'])
    result = {
        'known_stream': pd.DataFrame(snapshot_rows),
        'open_world_stream': pd.DataFrame(open_rows),
        'calibration_stream': pd.DataFrame(calibration_rows),
    }
    out_root = Path(local_cfg.output_dir) / f'seed_{seed}' / variant.replace(' ', '_')
    out_root.mkdir(parents=True, exist_ok=True)
    for name, df in result.items():
        df.to_csv(out_root / f'{name}.csv', index=False)
    return result

In [ ]:
# Run the primary seed.
# primary_results = run_single_seed_experiment(cfg.seed, variant='DriftGraph-BCP')

In [ ]:
# Run all baselines under the primary seed.
# baseline_results = {}
# for baseline_name in BASELINE_REGISTRY:
#     baseline_results[baseline_name] = run_single_seed_experiment(cfg.seed, variant=baseline_name)

In [ ]:
# Run the ablation suite under the primary seed.
# ablation_results = {}
# for ablation_name in ABLATION_REGISTRY:
#     ablation_results[ablation_name] = run_single_seed_experiment(cfg.seed, variant=ablation_name)

In [ ]:
# Run five-seed robustness.
# robustness_results = {}
# for seed in cfg.robustness_seeds:
#     robustness_results[seed] = {'DriftGraph-BCP': run_single_seed_experiment(seed, variant='DriftGraph-BCP')}
#     for baseline_name in BASELINE_REGISTRY:
#         robustness_results[seed][baseline_name] = run_single_seed_experiment(seed, variant=baseline_name)

## 15. Controlled drift stress suite

The stress suite evaluates covariate, noise, sparse-feature, class-prior, structural, and concept drift. The label-free BOCPD and delayed concept monitor are evaluated separately and jointly.

In [ ]:
def apply_controlled_drift(X: np.ndarray, y: np.ndarray, scenario: str, severity: float, rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    Xd = X.copy()
    yd = y.copy()
    if scenario == 'covariate':
        Xd = Xd + rng.normal(0, severity, Xd.shape).astype(np.float32)
    elif scenario == 'noise':
        Xd = Xd + rng.normal(0, severity * 1.5, Xd.shape).astype(np.float32)
    elif scenario == 'sparse_feature':
        n_features = Xd.shape[1]
        selected = rng.choice(n_features, size=max(1, int(0.10 * n_features)), replace=False)
        Xd[:, selected] = Xd[:, selected] + rng.normal(0, severity * 2.0, (Xd.shape[0], len(selected))).astype(np.float32)
    elif scenario == 'prior':
        pass
    elif scenario == 'structural':
        Xd = Xd + rng.normal(0, severity * 0.75, Xd.shape).astype(np.float32)
    elif scenario == 'concept':
        if len(np.unique(yd)) > 1:
            mask = rng.random(len(yd)) < severity
            classes = np.unique(yd)
            for i in np.where(mask)[0]:
                yd[i] = rng.choice(classes[classes != yd[i]])
    return Xd, yd


def run_controlled_drift_suite(runner: DriftGraphBCPRunner, X_reference: np.ndarray, y_reference: np.ndarray) -> pd.DataFrame:
    rng = np.random.default_rng(runner.cfg.seed)
    scenarios = ['concept', 'covariate', 'noise', 'prior', 'sparse_feature', 'structural']
    rows = []
    for scenario in scenarios:
        for seq in range(runner.cfg.controlled_drift_sequences):
            detector = StudentTBOCPD(hazard=0.05)
            detected = False
            delay = np.nan
            false_alarms = 0
            for batch in range(runner.cfg.controlled_drift_batches):
                idx = rng.choice(np.arange(X_reference.shape[0]), size=min(runner.cfg.stream_snapshot_size, X_reference.shape[0]), replace=True)
                Xb, yb = X_reference[idx], y_reference[idx]
                if batch >= runner.cfg.controlled_drift_position:
                    Xb, yb = apply_controlled_drift(Xb, yb, scenario, severity=0.35, rng=rng)
                score = float(np.mean(np.abs(Xb.mean(axis=0))))
                state = detector.step(score)
                alarm = state['short_run_mass'] > 0.65
                if alarm and batch < runner.cfg.controlled_drift_position:
                    false_alarms += 1
                if alarm and batch >= runner.cfg.controlled_drift_position and not detected:
                    detected = True
                    delay = batch - runner.cfg.controlled_drift_position
            rows.append({'scenario': scenario, 'sequence': seq, 'detected': int(detected), 'delay': delay, 'false_alarms': false_alarms})
    return pd.DataFrame(rows)

## 16. Summary table construction

These functions assemble the core manuscript tables from result artifacts generated by the primary, baseline, ablation, five-seed, and external experiments.

In [ ]:
def summarize_stream_results(result: Dict[str, pd.DataFrame]) -> Dict[str, float]:
    known = result.get('known_stream', pd.DataFrame())
    openw = result.get('open_world_stream', pd.DataFrame())
    cal = result.get('calibration_stream', pd.DataFrame())
    summary = {}
    if not known.empty:
        summary['known_macro_f1'] = float(known['macro_f1'].mean())
        summary['known_mcc'] = float(known['mcc'].mean())
        summary['known_auroc'] = float(known['macro_auroc'].mean()) if 'macro_auroc' in known else np.nan
        summary['known_auprc'] = float(known['macro_auprc'].mean()) if 'macro_auprc' in known else np.nan
    if not openw.empty:
        summary['unknown_auroc'] = float(openw['unknown_auroc'].mean())
        summary['unknown_f1'] = float(openw['unknown_f1'].mean())
        summary['false_known_rate'] = float(openw['false_known_rate'].mean())
    if not cal.empty:
        summary['ece'] = float(cal['ece'].mean())
        summary['mean_entropy'] = float(cal['mean_entropy'].mean())
        summary['mean_mi'] = float(cal['mean_mi'].mean())
        summary['mean_variance'] = float(cal['mean_variance'].mean())
    return summary


def build_baseline_comparison_table(primary: Dict[str, pd.DataFrame], baselines: Dict[str, Dict[str, pd.DataFrame]]) -> pd.DataFrame:
    rows = [{'model': 'DriftGraph-BCP', **summarize_stream_results(primary)}]
    for name, result in baselines.items():
        rows.append({'model': name, **summarize_stream_results(result)})
    return pd.DataFrame(rows)


def build_five_seed_table(seed_results: Dict[int, Dict[str, Dict[str, pd.DataFrame]]]) -> pd.DataFrame:
    records = []
    for seed, models in seed_results.items():
        for model, result in models.items():
            record = {'seed': seed, 'model': model}
            record.update(summarize_stream_results(result))
            records.append(record)
    long_df = pd.DataFrame(records)
    summary = []
    metrics = [c for c in long_df.columns if c not in {'seed', 'model'}]
    for model, group in long_df.groupby('model'):
        row = {'model': model}
        for metric in metrics:
            row[f'{metric}_mean'] = float(group[metric].mean())
            row[f'{metric}_sd'] = float(group[metric].std(ddof=1))
        summary.append(row)
    return pd.DataFrame(summary)

In [ ]:
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x); y = np.asarray(y)
    gt = sum(a > b for a in x for b in y)
    lt = sum(a < b for a in x for b in y)
    return float((gt - lt) / max(len(x) * len(y), 1))


def holm_adjust(p_values: List[float]) -> List[float]:
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        adj = min(1.0, (m - rank) * p_values[idx])
        running = max(running, adj)
        adjusted[idx] = running
    return adjusted.tolist()


def paired_statistical_tests(long_df: pd.DataFrame, proposed_model: str = 'DriftGraph-BCP') -> pd.DataFrame:
    metrics = [c for c in long_df.columns if c not in {'seed', 'model'}]
    records = []
    proposed = long_df[long_df['model'] == proposed_model].set_index('seed')
    for model in sorted(set(long_df['model']) - {proposed_model}):
        comp = long_df[long_df['model'] == model].set_index('seed')
        pvals = []
        temp = []
        for metric in metrics:
            common = proposed.index.intersection(comp.index)
            x = proposed.loc[common, metric].values
            y = comp.loc[common, metric].values
            diff = x - y
            if len(common) >= 3:
                try:
                    normal_p = stats.shapiro(diff).pvalue if len(common) >= 3 else 0.0
                    if normal_p > 0.05:
                        stat_p = stats.ttest_rel(x, y).pvalue
                        test = 'paired t-test'
                    else:
                        stat_p = stats.wilcoxon(x, y, zero_method='zsplit').pvalue
                        test = 'Wilcoxon signed-rank'
                except Exception:
                    stat_p = np.nan
                    test = 'not available'
            else:
                stat_p = np.nan
                test = 'not available'
            pvals.append(float(stat_p) if np.isfinite(stat_p) else 1.0)
            dz = float(np.mean(diff) / (np.std(diff, ddof=1) + 1e-12)) if len(diff) > 1 else np.nan
            temp.append({'comparator': model, 'metric': metric, 'test': test, 'p_value': stat_p, 'cohen_dz': dz, 'cliffs_delta': cliffs_delta(x, y), 'proposed_mean': float(np.mean(x)), 'comparator_mean': float(np.mean(y))})
        adj = holm_adjust(pvals)
        for rec, p_adj in zip(temp, adj):
            rec['holm_adjusted_p'] = p_adj
            records.append(rec)
    return pd.DataFrame(records)

## 17. Figure generation

The plotting functions draw the complete manuscript figure family from saved CSV artifacts. They intentionally use only Matplotlib to keep the environment simple and stable.

In [ ]:
def save_figure(fig, name: str) -> Path:
    fig_dir = Path(cfg.output_dir) / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    path = fig_dir / name
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return path


def plot_protocol_roles(role_summary: pd.DataFrame) -> Path:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.barh(role_summary['protocol_role'], role_summary['rows'])
    ax.set_xlabel('Rows')
    ax.set_title('Open-world protocol composition')
    ax.grid(axis='x', alpha=0.25)
    return save_figure(fig, 'protocol_roles.png')


def plot_phase_performance(phase_df: pd.DataFrame) -> Path:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
    x = np.arange(len(phase_df))
    labels = phase_df['phase_label'].tolist() if 'phase_label' in phase_df else phase_df.index.astype(str).tolist()
    for ax, col, title in zip(axes.ravel(), ['macro_f1', 'unknown_auroc', 'ece', 'conformal_coverage'], ['Macro-F1', 'Unknown AUROC', 'ECE', 'Conformal coverage']):
        if col in phase_df:
            ax.plot(x, phase_df[col], marker='o', linewidth=2)
            ax.set_xticks(x, labels, rotation=20)
            ax.set_title(title)
            ax.grid(alpha=0.25)
    return save_figure(fig, 'phase_performance.png')


def plot_reliability(bin_df: pd.DataFrame) -> Path:
    fig, ax = plt.subplots(figsize=(5.6, 5.0))
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Ideal calibration')
    ax.plot(bin_df['mean_confidence'], bin_df['empirical_accuracy'], marker='o', linewidth=2, label='DriftGraph-BCP')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Mean predicted confidence')
    ax.set_ylabel('Empirical accuracy')
    ax.set_title('Posterior-predictive reliability diagram')
    ax.legend()
    ax.grid(alpha=0.25)
    return save_figure(fig, 'reliability_diagram.png')


def plot_baseline_heatmap(compare_df: pd.DataFrame) -> Path:
    metrics = [c for c in ['known_macro_f1', 'unknown_auroc', 'ece', 'avg_forgetting'] if c in compare_df]
    mat = compare_df[metrics].values.astype(float)
    score = np.zeros_like(mat)
    for j, metric in enumerate(metrics):
        col = mat[:, j]
        if metric in {'ece', 'avg_forgetting'}:
            score[:, j] = (col.max() - col) / (col.max() - col.min() + 1e-12)
        else:
            score[:, j] = (col - col.min()) / (col.max() - col.min() + 1e-12)
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(score, aspect='auto', cmap='YlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(metrics)), metrics, rotation=20, ha='right')
    ax.set_yticks(range(len(compare_df)), compare_df['model'].tolist())
    fig.colorbar(im, ax=ax, label='Direction-aware score')
    ax.set_title('Direction-aware baseline comparison')
    return save_figure(fig, 'baseline_comparison_heatmap.png')

## 18. External replication

Run Edge-IIoTset with the same modeling logic after the primary protocol is frozen. Dataset-specific label resolution and feature selection are allowed, but no test-partition tuning should be performed.

In [ ]:
def run_external_replication() -> Dict[str, pd.DataFrame]:
    external_manifest = build_manifest(cfg.external_data_dir, cfg.external_dataset_name)
    external_df = read_csv_collection([Path(p) for p in external_manifest['path']])
    X_ext_raw, y_ext_raw, audit = feature_control(external_df)
    X_ext = feature_pipe.transform(X_ext_raw)
    roles_ext = assign_protocol_roles(y_ext_raw)
    partitions_ext = sample_role_partition(external_df, roles_ext, cfg.seed)
    runner = DriftGraphBCPRunner(cfg, variant='DriftGraph-BCP')
    train_rows = partitions_ext.loc[partitions_ext['partition'] == 'train', 'row_index'].values
    runner.fit_initial(X_ext, y_ext_raw, train_rows)
    stream_ext = build_stream_schedule(partitions_ext, cfg, cfg.seed)
    return {'external_manifest': external_manifest, 'external_stream': stream_ext}

# external_results = run_external_replication()

## 19. Reproducibility record

The final cell records configuration, package versions, dataset manifests, and artifact inventory.

In [ ]:
def package_versions() -> Dict[str, str]:
    versions = {
        'python': ' '.join(os.sys.version.splitlines()),
        'numpy': np.__version__,
        'pandas': pd.__version__,
    }
    try:
        import sklearn
        versions['scikit_learn'] = sklearn.__version__
    except Exception:
        pass
    if TORCH_AVAILABLE:
        versions['torch'] = torch.__version__
        versions['cuda_available'] = str(torch.cuda.is_available())
        versions['device'] = cfg.device
    return versions


def write_reproducibility_record(config: ProjectConfig, manifest: pd.DataFrame) -> Path:
    record = {
        'config': asdict(config),
        'versions': package_versions(),
        'manifest_rows': manifest.to_dict(orient='records'),
    }
    path = Path(config.output_dir) / 'reproducibility_record.json'
    with path.open('w', encoding='utf-8') as f:
        json.dump(record, f, indent=2)
    return path

reproducibility_record_path = write_reproducibility_record(cfg, primary_manifest)
reproducibility_record_path